In [1]:
## Set up the input datasets

import sys
import os 

import pandas as pd

import numpy as np 

import matplotlib.pyplot as plt 

import time as time

import cv2 
from PIL import Image, ImageFilter

import random

import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from torch import nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import OneCycleLR


import torchvision.transforms as T
from torchvision.transforms import InterpolationMode
from torchvision.models.segmentation import deeplabv3_mobilenet_v3_large, deeplabv3_resnet50
import torchvision.models as models

from tqdm import tqdm

from sklearn.model_selection import train_test_split

In [2]:
# Load the CSV file with labels
labels_df = pd.read_csv("../../train_final.csv")

# # Some images are broken/unreadable
# # Use these lines to only read in the "valid" images to stop 
with open('../../broken_images/valid_images_list.txt', 'r') as file:
    valid_image_paths = [line.strip() for line in file]

# with open('/kaggle/input/bhf-reference-files/valid_test_images_list.txt', 'r') as file:
#     valid_test_images = [line.strip() for line in file]

FileNotFoundError: [Errno 2] No such file or directory: '../../train_final.csv'

In [ ]:
mean_vals = labels_df[['STTC', 'HYP', 'MI', 'CD', 'AF']].to_numpy().mean(axis=0) # AF is quite rare, consider adding in class weighting
pos_weight = (1 - mean_vals)/mean_vals
pos_weight

array([ 3.12362637,  7.11351351,  2.93034826,  3.78482627, 13.53049371])

In [ ]:

# If you want the full path:
def get_full_png_filenames(directory):
    """Returns a list of full paths to .png files in a directory."""
    full_paths = []
    try:
        for filename in os.listdir(directory):
            if filename.endswith(".png"):
                full_path = os.path.join(directory, filename)  # Construct full path
                full_paths.append(full_path)
    except FileNotFoundError:
        print(f"Error: Directory '{directory}' not found.")
        return []
    except Exception as e:
        print(f"An error occurred: {e}")
        return []
    return full_paths

full_png_paths = get_full_png_filenames('../../data/train/')
full_png_paths_index = [path.split('/')[-1] for path in full_png_paths]

In [ ]:
full_train_paths, full_test_paths = train_test_split(valid_image_paths, test_size=0.1, random_state=42)

In [ ]:
train_paths = ['../../data/train/'+ path.split('/')[-1] for path in full_train_paths if path.split('/')[-1] in full_png_paths_index]
test_paths = ['../../data/train/'+ path.split('/')[-1] for path in full_test_paths if path.split('/')[-1] in full_png_paths_index]

## Defining Segmentation Part of the Model

In [ ]:
def generate_output(image: np.array, corners: np.ndarray, scale: tuple = None):
    """Generates a perspective-transformed output image."""
    corners = order_points(corners)  # Ensure correct point order

    if scale is not None:
        corners *= np.array(scale, dtype=np.float32)  # Vectorized multiplication

    destination_corners = find_dest(corners)  # Get transformation target points
    M = cv2.getPerspectiveTransform(corners, destination_corners)  # Perspective matrix

    max_width, max_height = map(int, destination_corners[2])  # Extract output size
    out = cv2.warpPerspective(image, M, (max_width, max_height), flags=cv2.INTER_LANCZOS4)

    return np.clip(out, 0, 255).astype(np.uint8)  # Ensure valid pixel values

def order_points(pts: np.ndarray) -> np.ndarray:
    """Rearranges points to: top-left, top-right, bottom-right, bottom-left."""
    pts = np.asarray(pts, dtype=np.float32)  # Ensure NumPy array
    s = pts.sum(axis=1)
    diff = np.diff(pts, axis=1)

    rect = np.zeros((4, 2), dtype=np.float32)
    rect[0] = pts[np.argmin(s)]  # Top-left
    rect[2] = pts[np.argmax(s)]  # Bottom-right
    rect[1] = pts[np.argmin(diff)]  # Top-right
    rect[3] = pts[np.argmax(diff)]  # Bottom-left

    return rect  # Already a NumPy array, no conversion needed

def find_dest(pts: np.ndarray) -> np.ndarray:
    """Computes destination points based on max width and height."""
    (tl, tr, br, bl) = pts  # Unpack ordered points

    # Compute max width and height
    width = int(max(np.linalg.norm(br - bl), np.linalg.norm(tr - tl)))
    height = int(max(np.linalg.norm(tr - br), np.linalg.norm(tl - bl)))

    return np.array([[0, 0], [width, 0], [width, height], [0, height]], dtype=np.float32)

def image_preprocess_transforms(mean=(0.4611, 0.4359, 0.3905), std=(0.2193, 0.2150, 0.2109)):
    common_transforms = T.Compose([T.ToTensor(), T.Normalize(mean, std),])
    return common_transforms

def get_model(model_path, device=None):
    checkpoints = torch.load(model_path, map_location=device, weights_only=True)
    model = deeplabv3_mobilenet_v3_large(num_classes=2, aux_loss=True).to(device)
    model.load_state_dict(checkpoints, strict=False)
    return model 

def deep_learning_scan(og_image: np.array = None, 
                       trained_model=None, 
                       image_size=384, 
                       BUFFER=10, 
                       preprocess_transforms=image_preprocess_transforms(), 
                       device='cpu'):

    half = image_size // 2
    imH, imW, C = og_image.shape
    image_model = cv2.resize(og_image, (image_size, image_size), interpolation=cv2.INTER_NEAREST)
    scale_x = imW / image_size
    scale_y = imH / image_size
    
    image_model = preprocess_transforms(image_model)
    image_model = torch.unsqueeze(image_model, dim=0)
    
    image_model = image_model.to('cpu')

    # # Device on CPU
    model_cpu = trained_model.to('cpu')
    model_cpu.eval()
    
    # Rest of your preprocessing remains the same, but use model_cpu
    with torch.no_grad():
        out = model_cpu(image_model)["out"]

    out = torch.argmax(out, dim=1, keepdims=True).permute(0, 2, 3, 1)[0].numpy().squeeze().astype(np.int32)
    r_H, r_W = out.shape

    out = np.pad(out * 255, pad_width=((half, half), (half, half)), mode='constant')

    # Edge Detection.
    canny = cv2.Canny(out.astype(np.uint8), 225, 255)
    canny = cv2.dilate(canny, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))
    contours, _ = cv2.findContours(canny, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)

    if not contours: 
        return None

    page = max(contours, key=cv2.contourArea)

    epsilon = 0.02 * cv2.arcLength(page, True)
    corners = cv2.approxPolyDP(page, epsilon, True)
    corners = np.concatenate(corners).astype(np.float32)

    corners[:, 0] -= half
    corners[:, 1] -= half
    corners[:, 0] *= scale_x
    corners[:, 1] *= scale_y
    
    if not (np.all(corners.min(axis=0) >= (0, 0)) and np.all(corners.max(axis=0) <= (imW, imH))):

        left_pad, top_pad, right_pad, bottom_pad = 0, 0, 0, 0

        box_corners = cv2.boxPoints(cv2.minAreaRect(corners.reshape((-1, 1, 2)))).astype(np.int32)

        box_x_min, box_y_min = np.min(box_corners, axis=0)
        box_x_max, box_y_max = np.max(box_corners, axis=0)

        # Find corner point which doesn't satify the image constraint
        # and record the amount of shift required to make the box
        # corner satisfy the constraint

        left_pad, right_pad = max(-box_x_min, 0) + BUFFER, max(box_x_max - imW, 0) + BUFFER
        top_pad, bottom_pad = max(-box_y_min, 0) + BUFFER, max(box_y_max - imH, 0) + BUFFER

        # # new image with additional zeros pixels
        # # adjust original image within the new 'image_extended'

        image_extended = np.pad(og_image, ((top_pad, bottom_pad), (left_pad, right_pad), (0, 0)), mode='constant')

        # shifting 'box_corners' the required amount
        box_corners[:, 0] += left_pad
        box_corners[:, 1] += top_pad

        corners = box_corners
        og_image = image_extended

    corners = sorted(corners.tolist())
    return generate_output(og_image, corners)

In [ ]:
seg_model_path = '../../model/object_detection/model_mbv3_iou_mix_2C049.pth'
seg_model = get_model(seg_model_path, 'cpu')

In [ ]:
class ECG_Dataset(Dataset):
    def __init__(self, image_paths, label_df, transforms=None, preprocess_fn=None, 
                 seg_model=None, device='cpu', new_width=2500, new_height=1200):
        self.image_paths = image_paths
        self.labels_df = label_df
        self.transforms = transforms
        self.preprocess_fn = preprocess_fn
        self.device = device
        self.local_model = seg_model
        self.new_width = new_width 
        self.new_height = new_height

    def __len__(self): 
        return len(self.image_paths)

    def __getitem__(self, idx):
        image_path = self.image_paths[idx]
        image = cv2.imread(image_path)
        if image is None:
            raise ValueError(f"Failed to load image: {image_path}")

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.preprocess_fn is not None:
            image = self.preprocess_fn(image, self.local_model)
            
        image = Image.fromarray(image)
        image = image.resize((self.new_width, self.new_height), Image.LANCZOS)
        
        if self.transforms is not None:
            image = self.transforms(image)
        
        # Extract label
        filename = os.path.basename(image_path)
        index = int(filename.rsplit('_', 1)[-1].split('.')[0])
        row = self.labels_df[self.labels_df['ID'] == index].iloc[:, 1:].values
        label = torch.tensor(row, dtype=torch.float32)

        return image, label

In [ ]:
class FastGaussianBlur(object):
    """Simplified Gaussian Blur using PIL's GaussianBlur filter"""
    def __init__(self, radius_range=(1, 3)):
        self.radius_min, self.radius_max = radius_range
    
    def __call__(self, img):
        radius = random.uniform(self.radius_min, self.radius_max)
        return img.filter(ImageFilter.GaussianBlur(radius=radius))

class FastPaperFoldEffect(object):
    """Simplified paper fold effect"""
    def __init__(self, max_intensity=0.3):
        self.max_intensity = max_intensity
    
    def __call__(self, img):
        img_tensor = T.ToTensor()(img)
        
        # Create single vertical or horizontal fold
        h, w = img_tensor.shape[1:]
        # is_vertical = random.random() > 0.5
        fold_pos = random.uniform(0.2, 0.8)

        fold_pos_px = int(w * fold_pos)
        img_tensor[:, :, fold_pos_px-2:fold_pos_px+2] *= random.uniform(0.7, 0.9)
        
        return T.ToPILImage()(img_tensor)

In [ ]:
# Optimized transform pipeline
train_transform = T.Compose([
    # Resize early to reduce computation
    T.Resize((224, 224), interpolation=InterpolationMode.LANCZOS),
    
    # Basic augmentations (fast)
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    T.RandomRotation(degrees=1),
    
    # Custom effects (simplified)
    T.RandomApply([FastGaussianBlur(radius_range=(1, 3))], p=0.2), # Blurring actually is quite rare in the data
    T.RandomApply([FastPaperFoldEffect(max_intensity=0.1)], p=0.2), # Randomly adds a fold to the paper 
    
    # Final transforms
    T.ToTensor(),
    T.Lambda(lambda x: x + torch.randn_like(x) * 0.01),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Validation transform
val_transform = T.Compose([
    T.Resize((224, 224), interpolation=InterpolationMode.LANCZOS),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
# Create the datasets for training and validation
train_dataset = ECG_Dataset(train_paths, labels_df, 
                            transforms=train_transform, 
                           preprocess_fn=deep_learning_scan, 
                           seg_model=seg_model)

val_dataset = ECG_Dataset(test_paths, labels_df, 
                          transforms=val_transform, 
                         preprocess_fn=deep_learning_scan, 
                         seg_model=seg_model)

# Create the dataloaders 
train_dataloader = DataLoader(train_dataset, 
                             num_workers = 0,
                             shuffle = True, 
                             batch_size = 32,
                             pin_memory=True)

val_dataloader = DataLoader(val_dataset, 
                             num_workers = 0,
                             shuffle = True, 
                             batch_size = 32,
                             pin_memory=True)

In [ ]:
# work out device 
device = torch.device("mps" if torch.backends.mps.is_available() \
                      else ("cuda" if torch.cuda.is_available() 
                            else "cpu"))

print(device)

mps


In [ ]:
# load model checkpoint 
model_path = "/kaggle/input/ecg_efficientnet/pytorch/default/1/model_checkpoint.pth"
checkpoint = torch.load(model_path, map_location=torch.device('cpu'), weights_only=False)

In [ ]:
# Load the EfficientNetV2-S model with pre-trained weights from ImageNet
model = models.efficientnet_v2_s(weights='DEFAULT')

num_ftrs = model.classifier[1].in_features  # Get input features of the classifier layer
model.classifier[1] = nn.Linear(num_ftrs, 5)  # Set output size to 5 (for your 5 classes)

model.to(device)

model.load_state_dict(checkpoint['model_state_dict'])

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 24, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): FusedMBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
        )
        (stochastic_depth): StochasticDepth(p=0.0, mode=row)
      )
      (1): FusedMBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
            (1): BatchNorm2d(24, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  

In [ ]:
def smooth_labels(labels, smoothing=0.1):
    """Apply label smoothing to multi-label targets."""
    return labels * (1 - smoothing) + 0.5 * smoothing

In [ ]:
pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float32).to(device)  

weighting = 0.75

# criterion1 = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
# criterion2 = nn.BCEWithLogitsLoss()

## Define focal loss

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=1, gamma=2, reduction='mean'):
        """
        Focal Loss for multi-label classification with support for label smoothing
        
        Args:
            alpha: Weighting factor for rare classes (can be a tensor for per-class weights)
            gamma: Focusing parameter - higher values put more focus on hard examples
            reduction: 'mean', 'sum', or 'none'
        """
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
        self.eps = 1e-6  # Small epsilon to prevent numerical instability
        
    def forward(self, inputs, targets):
        """
        Args:
            inputs: Model predictions (logits), shape (N, C)
            targets: Ground truth labels with optional smoothing, shape (N, C)
        """
        # Get device from inputs
        device = inputs.device
        
        # Apply sigmoid to get probabilities
        probs = torch.sigmoid(inputs)
        
        # Calculate BCE loss manually to handle soft labels
        # Add epsilon for numerical stability
        bce_loss = -(targets * torch.log(probs + self.eps) + 
                     (1 - targets) * torch.log(1 - probs + self.eps))
        
        # For focal weighting, calculate pt (probability of target)
        # This works with smoothed labels by interpolating between p and 1-p
        pt = targets * probs + (1 - targets) * (1 - probs)
        
        # Apply focusing parameter
        focal_weight = (1 - pt) ** self.gamma
        
        # Apply alpha weighting
        if isinstance(self.alpha, torch.Tensor):
            # Move alpha to the same device as inputs
            alpha = self.alpha.to(device)
            # Interpolate alpha weighting for soft labels
            alpha_weight = targets * alpha.unsqueeze(0) + (1 - targets) * (1 - alpha).unsqueeze(0)
        else:
            # Use scalar alpha
            alpha_weight = torch.ones_like(targets) * self.alpha
            
        # Calculate final loss
        loss = alpha_weight * focal_weight * bce_loss
        
        # Ensure loss is not negative due to numerical issues
        loss = torch.clamp(loss, min=0)
        
        # Apply reduction
        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        else:  # 'none'
            return loss

# Example usage with class-specific alpha for hypertrophy
def get_focal_loss_for_ecg(num_classes, hypertrophy_idx, hypertrophy_alpha=2.0, gamma=2.0):
    """
    Create a focal loss with higher weight for hypertrophy class
    
    Args:
        num_classes: Total number of label classes
        hypertrophy_idx: Index of the hypertrophy class
        hypertrophy_alpha: Alpha value for hypertrophy class
        gamma: Focusing parameter
    """
    alpha = torch.ones(num_classes) * 1.0  # Default alpha for all classes
    alpha[hypertrophy_idx] = hypertrophy_alpha  # Higher alpha for hypertrophy
    
    return FocalLoss(alpha=alpha, gamma=gamma)

criterion = get_focal_loss_for_ecg(
    num_classes=5,
    hypertrophy_idx=1,  # Index of your hypertrophy class
    hypertrophy_alpha=2.0,
    gamma=2.0
)

## Define LR Scheduler

In [ ]:
from torch.optim.lr_scheduler import _LRScheduler

In [ ]:
class WarmupCyclicCosineAnnealingLR(_LRScheduler):
    """
    Cyclic learning rate scheduler with gradual warm-up and cosine decay.
    
    Args:
        optimizer (Optimizer): Wrapped optimizer.
        cycle_length (int): Number of epochs in each cycle.
        max_lr (float): Maximum learning rate at the peak of the first cycle.
        min_lr (float): Absolute minimum learning rate to never go below.
        cycle_decay_factor (float): Factor that determines how low the lr goes at the end of each cycle 
                                   (relative to that cycle's max_lr).
        final_cycles_decay (float): Rate at which the max_lr decays each cycle.
        warmup_epochs (int): Number of warm-up epochs at the start of each cycle.
        start_epoch (int): The epoch from which to start/resume the scheduler.
        last_epoch (int): The index of the last epoch. Default: -1.
    """
    def __init__(self, optimizer, cycle_length=8, max_lr=1e-3, min_lr=1e-5, 
                 cycle_decay_factor=0.1, final_cycles_decay=0.85, 
                 warmup_epochs=2, start_epoch=0, last_epoch=-1):
        self.cycle_length = cycle_length
        self.max_lr = max_lr
        self.min_lr = min_lr
        self.cycle_decay_factor = cycle_decay_factor
        self.final_cycles_decay = final_cycles_decay
        self.warmup_epochs = warmup_epochs
        self.start_epoch = start_epoch
        
        # Initialize with last_epoch = start_epoch - 1 if we're starting from a specific epoch
        if start_epoch > 0 and last_epoch == -1:
            last_epoch = start_epoch - 1
            
        super(WarmupCyclicCosineAnnealingLR, self).__init__(optimizer, last_epoch)
        
        # # Set the initial learning rate based on the start_epoch
        # if start_epoch > 0:
        #     self.step()
        
    def get_lr(self):
        # Calculate the effective epoch (including the start_epoch offset)
        effective_epoch = self.last_epoch
        
        cycle = effective_epoch // self.cycle_length
        cycle_position = effective_epoch % self.cycle_length
        
        # Decaying max LR over cycles
        cycle_max_lr = max(
            self.max_lr * (self.final_cycles_decay ** cycle),
            self.min_lr
        )
        
        # Calculate the minimum LR for this cycle (factor * cycle_max_lr, but never below self.min_lr)
        cycle_min_lr = max(cycle_max_lr * self.cycle_decay_factor, self.min_lr)
        
        # In the first cycle, we might want to start at a different rate
        if cycle == 0 and cycle_position == 0:
            starting_lr = cycle_max_lr * self.cycle_decay_factor
        else:
            # For all subsequent cycles, we start at cycle_min_lr
            starting_lr = cycle_min_lr
        
        # Warm-up phase
        if cycle_position < self.warmup_epochs:
            # Warm up from the starting_lr to cycle_max_lr
            warmup_factor = cycle_position / self.warmup_epochs
            lr = starting_lr + (cycle_max_lr - starting_lr) * warmup_factor
        else:
            # Cosine decay during the remaining epochs of the cycle from cycle_max_lr to cycle_min_lr
            decay_phase = cycle_position - self.warmup_epochs
            decay_length = self.cycle_length - self.warmup_epochs
            cosine_factor = 0.5 * (1 + np.cos(np.pi * (decay_phase / decay_length)))
            lr = cycle_min_lr + (cycle_max_lr - cycle_min_lr) * cosine_factor
        
        return [lr for _ in self.base_lrs]
        
def plot_lr_schedule(scheduler, optimizer, epochs=100):
    """Plot the learning rate schedule."""
    lrs = []
    for epoch in range(epochs):
        lrs.append(optimizer.param_groups[0]['lr'])
        scheduler.step()
    
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(lrs, '-x')
    ax.set_yscale("log")
    ax.set_xlabel('Epochs')
    ax.set_ylabel('Learning Rate')
    ax.set_title('Cyclic Learning Rate Schedule')
    ax.grid(True)
    ax.set_ylim(0, max(lrs) * 1.1)
    plt.show()

In [ ]:
num_epochs = 20
current_epoch = checkpoint['epoch']

In [ ]:
initial_lr = 1.0e-3
optimizer = optim.AdamW(
    model.parameters(), 
    lr=initial_lr,
    weight_decay=1e-4
)

optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

# Create the scheduler
scheduler = WarmupCyclicCosineAnnealingLR(
    optimizer,
    cycle_length=14,           # 8 epochs per cycle
    max_lr=initial_lr,        # max_lr is the initial_lr
    min_lr=1e-5,              # absolute minimum lr to never go below
    cycle_decay_factor=0.1,   # cycle drops to 10% of its max_lr
    warmup_epochs=4,          # how many epochs to warm up
    final_cycles_decay=0.7, # decay rate for max_lr each cycle
    start_epoch=current_epoch, 
    last_epoch=current_epoch-1
)

scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

In [ ]:
from sklearn.metrics import f1_score, hamming_loss  # Evaluation metrics

def calculate_metrics(outputs, labels):
    """Calculate multiple metrics for multi-label classification"""
    # Convert outputs to predictions (0 or 1)
    predictions = (torch.sigmoid(outputs) > 0.5).float()
    
    # Move tensors to CPU and convert to numpy for sklearn metrics
    predictions_np = predictions.cpu().numpy()
    labels_np = labels.cpu().numpy()
    
    # Exact match accuracy (all labels must match)
    exact_match = (predictions == labels).all(dim=1).float().mean().item()
    
    # Hamming loss (fraction of labels that are incorrectly predicted)
    hamming = hamming_loss(labels_np, predictions_np)
    
    # Sample-wise F1 score with zero_division handling
    try:
        f1 = f1_score(labels_np, predictions_np, average='samples', zero_division=0)
    except:
        f1 = 0.0  # or np.nan if you prefer to track when this happens
    
    # Per-class F1 scores with zero_division handling
    try:
        per_class_f1 = f1_score(labels_np, predictions_np, average=None, zero_division=0)
    except:
        per_class_f1 = np.zeros(labels_np.shape[1])  # or np.full(labels_np.shape[1], np.nan)
    
    return {
        'exact_match': exact_match,
        'hamming_loss': hamming,
        'f1_score': f1,
        'per_class_f1': per_class_f1
    }

In [ ]:
# Define where to save the model
model_save_path = "model_checkpoint.pth"

running_outputs = []

for epoch in range(current_epoch, num_epochs):
    # Training phase
    model.train()
    running_train_loss = 0.0
    train_metrics = {
        'exact_match': 0.0,
        'hamming_loss': 0.0,
        'f1_score': 0.0,
        'per_class_f1': np.zeros(5)  # Assuming 5 classes
    }
    
    for inputs, labels in tqdm(train_dataloader, desc=f"Epoch {epoch+1} - Training", unit="batch"):
        inputs, labels = inputs.to(device), labels.to(device)
        labels = labels.squeeze(1)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        smoothed_labels = smooth_labels(labels)
        loss = criterion(outputs, smoothed_labels)
        loss.backward()
        optimizer.step()
        
        # Calculate and accumulate metrics
        batch_metrics = calculate_metrics(outputs, labels)
        running_train_loss += loss.item()
        train_metrics['exact_match'] += batch_metrics['exact_match']
        train_metrics['hamming_loss'] += batch_metrics['hamming_loss']
        train_metrics['f1_score'] += batch_metrics['f1_score']
        train_metrics['per_class_f1'] += batch_metrics['per_class_f1']

    scheduler.step()
    
    # Calculate average metrics for training
    num_batches = len(train_dataloader)
    avg_train_loss = running_train_loss / num_batches
    train_metrics = {k: v / num_batches for k, v in train_metrics.items()}
    
    print(f"\nEpoch [{epoch+1}/{num_epochs}] Training Metrics:")
    print(f"Loss: {avg_train_loss:.4f}")
    print(f"Exact Match Accuracy: {train_metrics['exact_match']:.4f}")
    print(f"Hamming Loss: {train_metrics['hamming_loss']:.4f}")
    print(f"F1 Score: {train_metrics['f1_score']:.4f}")
    print("Per-class F1 Scores:", ' '.join(f"{x:.4f}" for x in train_metrics['per_class_f1']))
    
    # Validation phase
    model.eval()
    running_val_loss = 0.0
    val_metrics = {
        'exact_match': 0.0,
        'hamming_loss': 0.0,
        'f1_score': 0.0,
        'per_class_f1': np.zeros(5)  # Assuming 5 classes
    }
    
    with torch.no_grad():
        for inputs, labels in tqdm(val_dataloader, desc=f"Epoch {epoch+1} - Validation", unit="batch"):
            inputs, labels = inputs.to(device), labels.to(device)
            labels = labels.squeeze(1)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            # Calculate and accumulate metrics
            batch_metrics = calculate_metrics(outputs, labels)
            running_val_loss += loss.item()
            val_metrics['exact_match'] += batch_metrics['exact_match']
            val_metrics['hamming_loss'] += batch_metrics['hamming_loss']
            val_metrics['f1_score'] += batch_metrics['f1_score']
            val_metrics['per_class_f1'] += batch_metrics['per_class_f1']
    
    # Calculate average metrics for validation
    num_val_batches = len(val_dataloader)
    avg_val_loss = running_val_loss / num_val_batches
    val_metrics = {k: v / num_val_batches for k, v in val_metrics.items()}
    
    print(f"\nEpoch [{epoch+1}/{num_epochs}] Validation Metrics:")
    print(f"Loss: {avg_val_loss:.4f}")
    print(f"Exact Match Accuracy: {val_metrics['exact_match']:.4f}")
    print(f"Hamming Loss: {val_metrics['hamming_loss']:.4f}")
    print(f"F1 Score: {val_metrics['f1_score']:.4f}")
    print("Per-class F1 Scores:", ' '.join(f"{x:.4f}" for x in val_metrics['per_class_f1']))
    
    # Save checkpoint with metrics

    checkpoint = {
    "epoch": epoch + 1,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "scheduler_state_dict": scheduler.state_dict(),
    "train_loss": avg_train_loss,
    "val_loss": avg_val_loss,
    "train_metrics": train_metrics,
    "val_metrics": val_metrics
    }
    
    torch.save(checkpoint, model_save_path)


Epoch 1 - Training: 100%|██████████| 142/142 [27:24<00:00, 11.58s/batch] 



Epoch [1/15] Training Metrics:
Loss: 1.2690
Exact Match Accuracy: 0.0150
Hamming Loss: 0.5665
F1 Score: 0.2147
Per-class F1 Scores: 0.3129 0.2010 0.3764 0.3103 0.1336


Epoch 1 - Validation: 100%|██████████| 15/15 [04:55<00:00, 19.67s/batch]



Epoch [1/15] Validation Metrics:
Loss: 1.1731
Exact Match Accuracy: 0.0731
Hamming Loss: 0.4117
F1 Score: 0.2059
Per-class F1 Scores: 0.2329 0.2398 0.4029 0.2709 0.0655


Epoch 2 - Training: 100%|██████████| 142/142 [28:57<00:00, 12.23s/batch] 


...


Epoch [14/15] Validation Metrics:
Loss: 0.7493
Exact Match Accuracy: 0.4531
Hamming Loss: 0.1746
F1 Score: 0.3868
Per-class F1 Scores: 0.6427 0.4758 0.6655 0.6885 0.4851


Epoch 15 - Training: 100%|██████████| 142/142 [23:23<00:00,  9.88s/batch]



Epoch [15/15] Training Metrics:
Loss: 0.7435
Exact Match Accuracy: 0.6448
Hamming Loss: 0.0908
F1 Score: 0.4592
Per-class F1 Scores: 0.8304 0.7097 0.8137 0.8173 0.5553


Epoch 15 - Validation: 100%|██████████| 15/15 [02:18<00:00,  9.25s/batch]



Epoch [15/15] Validation Metrics:
Loss: 0.7579
Exact Match Accuracy: 0.4944
Hamming Loss: 0.1454
F1 Score: 0.3958
Per-class F1 Scores: 0.6681 0.5482 0.7095 0.7144 0.5141
